In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/TwoD_CEUS_test
/Users/samantha/QuantUS-Plugins-CEUS


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [24]:
scan_type = 'nifti'

scan_path = '/Volumes/Extreme Pro/UCSD_SSD/ChinaData/3D-034/1st/CEUS-26152-1.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [25]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [26]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [55]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/necrosis_testing/p34/v1/voi_necrotic_removed.nii.gz'
seg_loader_kwargs = {}

In [56]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis

In [57]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [58]:
analysis_type = 'curves_paramap'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [59]:
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['sag_vox_ovrlp', 'cor_vox_ovrlp', 'cor_vox_len', 'ax_vox_len', 'ax_vox_ovrlp', 'sag_vox_len']


In [60]:
# Set frame rate (adjust this value to match your actual video fps)
image_data.frame_rate = 1  # e.g., 30 fps

analysis_kwargs = {
    'ax_vox_ovrlp': 50,
    'sag_vox_ovrlp': 50,
    'cor_vox_ovrlp': 50,
    'ax_vox_len': 20.0,
    'sag_vox_len': 20.0,
    'cor_vox_len': 20.0,
}


In [ ]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves:  13%|█▎        | 25/197 [00:17<02:02,  1.40it/s]

## Curve Quantification

In [ ]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [ ]:
function_names = [] # Empty list will use all functions
output_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/output/curve_quant_raw.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [ ]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

In [ ]:
import numpy as np
data = curve_quant.data_dict[0]
print(f"{'Parameter':<30} {'Value':>15}")
print("-" * 47)
for key, value in data.items():
    if isinstance(value, float):
        print(f"{key:<30} {value:>15.4f}")
    else:
        print(f"{key:<30} {str(value):>15}")

# Volume calculation
pixdim = seg_data.pixdim
voxel_vol_mm3 = np.prod(pixdim)
n_voxels = int(np.sum(seg_data.seg_mask > 0))
vol_mm3 = n_voxels * voxel_vol_mm3

print(f"\n{'VOI Volume':<30}")
print("-" * 47)
print(f"{'Voxel spacing (mm)':<30} {str(pixdim):>15}")
print(f"{'Voxels in VOI':<30} {n_voxels:>15,}")
print(f"{'Volume (mm³)':<30} {vol_mm3:>15.1f}")
print(f"{'Volume (cm³)':<30} {vol_mm3/1000:>15.2f}")

Parameter                                Value
-----------------------------------------------
Scan Name                      CEUS-26152-1.nii
Segmentation Name              v1_nonecrosis_voi
Window-Axial Start Pix                       3
Window-Sagittal Start Pix                   93
Window-Axial End Pix                        14
Window-Sagittal End Pix                    105
Window-Coronal Start Pix                    49
Window-Coronal End Pix                      55
AUC_NoFit_TIC                         135.9379
AUC_full_TIC                           77.4190
PE_full_TIC                             0.6333
TP_full_TIC                            53.1328
MTT_full_TIC                          113.7165
T0_full_TIC                             0.0000
Mu_full_TIC                             4.4801
Sigma_full_TIC                          0.7122
PE_Ix_full_TIC                              53
AUC_select_TIC                         77.4190
PE_select_TIC                           0.6333
TP_select

## Save Results to CSV

In [ ]:
import pandas as pd
import os
from datetime import datetime

out_path = "/Users/samantha/Desktop/ultrasound lab stuff/china data/necrosis_testing/p34/output.csv"

data = curve_quant.data_dict[0]
df = pd.DataFrame([data])
df["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

scan_col = "Scan Name"

if os.path.exists(out_path):
    existing = pd.read_csv(out_path)
    if "Timestamp" not in existing.columns:
        existing["Timestamp"] = pd.NaT
        existing.to_csv(out_path, index=False)
    df.to_csv(out_path, mode="a", header=False, index=False)
    print(f"Appended to {out_path}")
else:
    df.to_csv(out_path, index=False)
    print(f"Created {out_path}")